# Los Angeles Crime Dataset — 01 · Limpeza

**Projeto Integrador III — Ciência de Dados e Machine Learning, CEUB**

Primeiro notebook da cadeia de três:

`01 limpeza` → **`data/processed/lapd_clean.csv`** → `02 clusterização` → `data/processed/amostra_clusters.csv` → `03 visualizações`

Este notebook baixa a base bruta do portal de dados abertos de Los Angeles (*Crime Data from 2020 to Present*), faz o tratamento e grava o CSV que o notebook de clusterização lê. **Rode os três notebooks a partir da mesma pasta de trabalho**, para que `data/raw` e `data/processed` sejam compartilhados.

The CSV file below is a dataset of crime incidents in the City of Los Angeles from 2020 to present, and we aim to clean it and explore it (EDA).

---

# 1) ELT (Extract, Load and Transform)

## 1.1) Extract

Firstly, We'll prepare the libraries and CSV file first:

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency, f_oneway

In [2]:
import os
import urllib.request

CAMINHO_BRUTO = "data/raw/Crime_Data_from_2020_to_Present.csv"
URL_CSV = "https://data.lacity.org/api/views/2nrs-mtv8/rows.csv?accessType=DOWNLOAD"
DIR_PROCESSADOS = "data/processed"
CAMINHO_LIMPO = f"{DIR_PROCESSADOS}/lapd_clean.csv"
DATA_CORTE = "2024-02-29"   # quebra estrutural: a composição dos registros muda após a migração para o NIBRS

os.makedirs(os.path.dirname(CAMINHO_BRUTO), exist_ok=True)
os.makedirs(DIR_PROCESSADOS, exist_ok=True)

if not os.path.exists(CAMINHO_BRUTO):
    print("Baixando a base…")
    urllib.request.urlretrieve(URL_CSV, CAMINHO_BRUTO)

df = pd.read_csv(CAMINHO_BRUTO)
df.columns = df.columns.str.strip()

Baixando a base…


Now We need to see it head and tail as an data frame, let's take a look:

In [3]:
# Display the dataframe data showing all the columns mandatorily
with pd.option_context('display.max_columns', None):
    display(df)

,DR_NO,Date Rptd,DATE OCC,TIME OCC,AREA,AREA NAME,Rpt Dist No,Part 1-2,Crm Cd,Crm Cd Desc,Mocodes,Vict Age,Vict Sex,Vict Descent,Premis Cd,Premis Desc,Weapon Used Cd,Weapon Desc,Status,Status Desc,Crm Cd 1,Crm Cd 2,Crm Cd 3,Crm Cd 4,LOCATION,Cross Street,LAT,LON
0,211507896,04/11/2021 12:00:00 AM,11/07/2020 12:00:00 AM,845,15,N Hollywood,1502,2,354,THEFT OF IDENTITY,0377,31,M,H,501.0,SINGLE FAMILY DWELLING,NaN,NaN,IC,Invest Cont,354.0,NaN,NaN,NaN,7800 BEEMAN AV,NaN,34.2124,-118.4092
1,201516622,10/21/2020 12:00:00 AM,10/18/2020 12:00:00 AM,1845,15,N Hollywood,1521,1,230,"ASSAULT WITH DEADLY WEAPON, AGGRAVATED ASSAULT",0416 0334 2004 1822 1414 0305 0319 0400,32,M,H,102.0,SIDEWALK,200.0,KNIFE WITH BLADE 6INCHES OR LESS,IC,Invest Cont,230.0,NaN,NaN,NaN,ATOLL AV,N GAULT,34.1993,-118.4203
2,240913563,12/10/2024 12:00:00 AM,10/30/2020 12:00:00 AM,1240,9,Van Nuys,933,2,354,THEFT OF IDENTITY,0377,30,M,W,501.0,SINGLE FAMILY DWELLING,NaN,NaN,IC,Invest Cont,354.0,NaN,NaN,NaN,14600 SYLVAN ST,NaN,34.1847,-118.4509
3,210704711,12/24/2020 12:00:00 AM,12/24/2020 12:00:00 AM,1310,7,Wilshire,782,1,331,THEFT FROM MOTOR VEHICLE - GRAND ($950.01 AND ...,0344,47,F,A,101.0,STREET,NaN,NaN,IC,Invest Cont,331.0,NaN,NaN,NaN,6000 COMEY AV,NaN,34.0339,-118.3747
4,201418201,10/03/2020 12:00:00 AM,09/29/2020 12:00:00 AM,1830,14,Pacific,1454,1,420,THEFT FROM MOTOR VEHICLE - PETTY ($950 & UNDER),1300 0344 1606 2032,63,M,H,103.0,ALLEY,NaN,NaN,IC,Invest Cont,420.0,NaN,NaN,NaN,4700 LA VILLA MARINA,NaN,33.9813,-118.4350
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1004889,240710284,07/24/2024 12:00:00 AM,07/23/2024 12:00:00 AM,1400,7,Wilshire,788,1,510,VEHICLE - STOLEN,NaN,0,NaN,NaN,101.0,STREET,NaN,NaN,IC,Invest Cont,510.0,NaN,NaN,NaN,4000 W 23RD ST,NaN,34.0362,-118.3284
1004890,240104953,01/15/2024 12:00:00 AM,01/15/2024 12:00:00 AM,100,1,Central,101,2,745,VANDALISM - MISDEAMEANOR ($399 OR UNDER),0329 0400 0416,0,X,X,503.0,HOTEL,500.0,UNKNOWN WEAPON/OTHER WEAPON,IC,Invest Cont,745.0,NaN,NaN,NaN,1300 W SUNSET BL,NaN,34.0685,-118.2460
1004891,240410786,10/14/2024 12:00:00 AM,10/11/2024 12:00:00 AM,2330,4,Hollenbeck,421,1,341,"THEFT-GRAND ($950.01 & OVER)EXCPT,GUNS,FOWL,LI...",0344,29,M,B,210.0,RESTAURANT/FAST FOOD,NaN,NaN,IC,Invest Cont,341.0,NaN,NaN,NaN,1700 ALBION ST,NaN,34.0675,-118.2240
1004892,240309674,04/24/2024 12:00:00 AM,04/24/2024 12:00:00 AM,1500,3,Southwest,358,1,230,"ASSAULT WITH DEADLY WEAPON, AGGRAVATED ASSAULT",1822 0334 0416 0445 0449 1202,70,F,W,102.0,SIDEWALK,308.0,STICK,IC,Invest Cont,230.0,NaN,NaN,NaN,FLOWER ST,JEFFERSON BL,34.0215,-118.2868


Let's get the shape:

In [4]:
df.shape

(1004894, 28)

Given that, We'll discover the meaning of each column, let's get some information about this data:

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1004894 entries, 0 to 1004893
Data columns (total 28 columns):
 #   Column          Non-Null Count    Dtype  
---  ------          --------------    -----  
 0   DR_NO           1004894 non-null  int64  
 1   Date Rptd       1004894 non-null  object 
 2   DATE OCC        1004894 non-null  object 
 3   TIME OCC        1004894 non-null  int64  
 4   AREA            1004894 non-null  int64  
 5   AREA NAME       1004894 non-null  object 
 6   Rpt Dist No     1004894 non-null  int64  
 7   Part 1-2        1004894 non-null  int64  
 8   Crm Cd          1004894 non-null  int64  
 9   Crm Cd Desc     1004894 non-null  object 
 10  Mocodes         853296 non-null   object 
 11  Vict Age        1004894 non-null  int64  
 12  Vict Sex        860263 non-null   object 
 13  Vict Descent    860251 non-null   object 
 14  Premis Cd       1004878 non-null  float64
 15  Premis Desc     1004306 non-null  object 
 16  Weapon Used Cd  327216 non-null   fl

# Dicionário de Dados de Crimes - LAPD

Abaixo está a descrição dos rótulos de colunas do conjunto de dados de crimes do Departamento de Polícia de Los Angeles (LAPD).

---

### **DR_NO** *(Division of Records Number)*
Número oficial do registro policial, composto por 2 dígitos referentes ao ano, ID da área e um código numérico de 5 dígitos.

### **Date Rptd**
Data em que o crime foi relatado à polícia (`MM/DD/AAAA`).

### **DATE OCC**
Data em que o crime ocorreu (`MM/DD/AAAA`).

### **TIME OCC**
Horário da ocorrência em formato militar de 24 horas (ex.: `1430` para 14:30).

### **AREA**
Código numérico sequencial (de 1 a 21) correspondente à divisão policial comunitária do LAPD (*Community Police Stations / Geographic Areas*).

### **AREA NAME**
Nome da divisão policial ou patrulha geográfica, fazendo referência a um ponto de referência ou à comunidade atendida.  
*Exemplo:* A divisão `77th Street` fica no cruzamento da South Broadway com a 77th Street, atendendo bairros da região sul de Los Angeles.

### **Rpt Dist No** *(Reporting District)*
Código de 4 dígitos que representa uma subárea dentro de uma Área Geográfica. Todos os registros de crimes referenciam essa sigla ("RD") para comparações estatísticas.

### **Part 1-2**
Categoria da gravidade do crime segundo a classificação estatística:
- **Parte 1:** Crimes mais graves (homicídio, roubo, furto, etc.).
- **Parte 2:** Infrações e contravenções de menor gravidade.

### **Crm Cd** *(Crime Code)*
Código numérico que indica o tipo de crime cometido.

### **Crm Cd Desc** *(Crime Code Description)*
Descrição detalhada referente ao código do crime fornecido.

### **Mocodes Modus Operandi**
Códigos numéricos associados às ações, métodos e comportamentos do suspeito durante a execução do crime (*Modus Operandi*).

### **Vict Age**
Idade da vítima registrada em anos (formato numérico de 2 dígitos).

### **Vict Sex**
Sexo biológico/identificação da vítima:
- `F`: Feminino (*Female*)
- `M`: Masculino (*Male*)
- `X`: Desconhecido (*Unknown*)

### **Vict Descent**
Código de origem etnia/descendência da vítima:
- `A`: Outro Asiático (*Other Asian*)
- `B`: Negro (*Black*)
- `C`: Chinês (*Chinese*)
- `D`: Cambojano (*Cambodian*)
- `F`: Filipino (*Filipino*)
- `G`: Guamês (*Guamanian*)
- `H`: Hispânico/Latino/Mexicano (*Hispanic/Latin/Mexican*)
- `I`: Indígena Americano/Nativo do Alasca (*American Indian/Alaskan Native*)
- `J`: Japonês (*Japanese*)
- `K`: Coreano (*Korean*)
- `L`: Laosiano (*Laotian*)
- `O`: Outro (*Other*)
- `P`: Ilhas do Pacífico (*Pacific Islander*)
- `S`: Samoano (*Samoan*)
- `U`: Havaiano (*Hawaiian*)
- `V`: Vietnamita (*Vietnamese*)
- `W`: Branco (*White*)
- `X`: Desconhecido (*Unknown*)
- `Z`: Indiano (*Asian Indian*)

### **Premis Cd** *(Premise Code)*
Código do tipo de estrutura, veículo ou local onde o crime ocorreu.

### **Premis Desc** *(Premise Description)*
Descrição detalhada do local da ocorrência.

### **Weapon Used Cd**
Código referente ao tipo de arma utilizada na prática do crime.

### **Weapon Desc**
Descrição detalhada do tipo de arma utilizada.

### **Status**
Status atual da investigação policial. O valor padrão é `IC` (*Investigation Continued* / Investigação em andamento).

### **Status Desc**
Descrição detalhada do código de status do caso.

### **Crm Cd 1**
Código do crime principal e mais grave cometido na ocorrência (idêntico ao `Crm Cd`). Códigos de menor valor numérico de classe indicam maior gravidade.

### **Crm Cd 2**
Código opcional para um segundo crime associado à ocorrência, de gravidade inferior ao Crime 1.

### **Crm Cd 3**
Código opcional para um terceiro crime associado à ocorrência, de gravidade inferior ao Crime 1.

### **Crm Cd 4**
Código opcional para um quarto crime associado à ocorrência, de gravidade inferior ao Crime 1.

### **LOCATION**
Endereço da ocorrência arredondado para a quadra/bloco de 100 mais próximo, mantendo o anonimato da localização exata.

### **Cross Street**
Rua de cruzamento próxima ao endereço registrado.

### **LAT**
Coordenada geográfica de Latitude.

### **LON**
Coordenada geográfica de Longitude.

## 1.2) Transform — limpeza

Regras aplicadas (e por quê):

| # | Tratamento | Motivo |
|---|---|---|
| 1 | Datas viram `datetime`; `TIME OCC` (HHMM) vira hora válida em `DATETIME OCC` | `DATE OCC` é mantida como data pura, porque horas inválidas (`NaT`) não podem apagar a data |
| 2 | **Corte em `DATA_CORTE` (29/02/2024)** | Quebra estrutural da série (migração para o NIBRS) |
| 3 | Duplicatas de `DR_NO` removidas | `DR_NO` identifica o registro |
| 4 | `Vict Age` ≤ 0 ou > 100 → ausente; `LAT`/`LON` = 0 → ausente | São valores sentinela, não medidas |
| 5 | **Nenhuma linha é descartada por ausência estrutural.** Sem sexo/ascendência → `NAO_APLICAVEL`; `X`/`H`/`-` → `DESCONHECIDO`; local ausente → `NAO_INFORMADO` | O `dropna` anterior eliminava ~15% da base e quase todos os `VEHICLE - STOLEN` (que não têm vítima pessoa) |
| 6 | `Weapon Used Cd` **continua ausente** quando não há arma | Preencher com 0 faria o código cair na faixa de armas de fogo (< 200) na clusterização |
| 7 | `Crm Cd 1–4` viram `Crm_Count`; colunas redundantes ou quase únicas são removidas | `LOCATION`, `Cross Street`, `Status Desc`, `TIME OCC` |
| 8 | Colunas derivadas `Week_OCC` e `shift_OCC` | Dia da semana e turno |

As colunas de descrição (`Crm Cd Desc`, `Premis Desc`, `Weapon Desc`, `AREA NAME`) **são mantidas**: a clusterização precisa delas para rotular crime, local e área.

In [6]:
df_clean = df.copy()

# Espaços em branco nas colunas de texto
for c in df_clean.select_dtypes(include=["object", "string"]).columns:
    df_clean[c] = df_clean[c].str.strip()

n_inicial = len(df_clean)
print(f"Registros brutos: {n_inicial:,}")

Registros brutos: 1,004,894


In [7]:
# ---------- 1) Datas ----------
def para_data(s):
    d = pd.to_datetime(s, format="%m/%d/%Y %I:%M:%S %p", errors="coerce")
    if d.isna().mean() > 0.5:                 # o portal já mudou o formato de data antes
        d = pd.to_datetime(s, format="mixed", errors="coerce")
    return d

df_clean["Date Rptd"] = para_data(df_clean["Date Rptd"])
df_clean["DATE OCC"] = para_data(df_clean["DATE OCC"]).dt.normalize()

sem_data = df_clean["DATE OCC"].isna().sum()
df_clean = df_clean.dropna(subset=["DATE OCC"])
print(f"Registros sem data de ocorrência válida (removidos): {sem_data:,}")

# TIME OCC está em HHMM. Horas inválidas (ex.: 2500, 1275) viram NaT apenas no DATETIME OCC.
t = pd.to_numeric(df_clean["TIME OCC"], errors="coerce")
hora_valida = t.between(0, 2359) & ((t % 100) < 60)
minutos = (t // 100) * 60 + (t % 100)
df_clean["DATETIME OCC"] = (df_clean["DATE OCC"] + pd.to_timedelta(minutos, unit="m")).where(hora_valida)
print(f"Horários inválidos (DATETIME OCC = NaT): {df_clean['DATETIME OCC'].isna().sum():,}")

# Diagnóstico: ocorrência registrada depois do relato (inconsistência de data, mantida e apenas sinalizada)
inconsistentes = (df_clean["DATE OCC"] > df_clean["Date Rptd"]).sum()
print(f"Registros com DATE OCC posterior a Date Rptd (apenas sinalizados): {inconsistentes:,}")

Registros sem data de ocorrência válida (removidos): 0
Horários inválidos (DATETIME OCC = NaT): 0
Registros com DATE OCC posterior a Date Rptd (apenas sinalizados): 0


In [8]:
# ---------- 2) Corte em DATA_CORTE ----------
antes = len(df_clean)
df_clean = df_clean[df_clean["DATE OCC"] <= pd.Timestamp(DATA_CORTE)].copy()
print(f"Corte em {DATA_CORTE}: {antes:,} → {len(df_clean):,} registros ({antes - len(df_clean):,} removidos)")

# ---------- 3) Duplicatas ----------
antes = len(df_clean)
df_clean = df_clean.drop_duplicates(subset="DR_NO", keep="first")
print(f"Duplicatas de DR_NO removidas: {antes - len(df_clean):,}")

Corte em 2024-02-29: 1,004,894 → 913,647 registros (91,247 removidos)
Duplicatas de DR_NO removidas: 0


In [9]:
# ---------- 4) Valores sentinela ----------
idade = pd.to_numeric(df_clean["Vict Age"], errors="coerce")
print(f"Vict Age ≤ 0 ou > 100 (viram ausente): {(~idade.between(1, 100) & idade.notna()).sum():,}")
df_clean["Vict Age"] = idade.where(idade.between(1, 100))

zero_geo = ((df_clean["LAT"] == 0) | (df_clean["LON"] == 0))
print(f"LAT/LON = 0 (viram ausente): {zero_geo.sum():,}")
df_clean.loc[zero_geo, ["LAT", "LON"]] = np.nan

Vict Age ≤ 0 ou > 100 (viram ausente): 229,906
LAT/LON = 0 (viram ausente): 2,240


In [10]:
# ---------- 5) Ausências estruturais viram categorias explícitas ----------
sexo = df_clean["Vict Sex"]
df_clean["Vict Sex"] = np.where(sexo.isna(), "NAO_APLICAVEL",
                                np.where(sexo.isin(["M", "F"]), sexo, "DESCONHECIDO"))   # X, H, '-'

desc = df_clean["Vict Descent"]
df_clean["Vict Descent"] = np.where(desc.isna(), "NAO_APLICAVEL",
                                    np.where(desc.isin(["X", "-", ""]), "DESCONHECIDO", desc))

df_clean["Premis Desc"] = df_clean["Premis Desc"].fillna("NAO_INFORMADO")
df_clean["Status"] = df_clean["Status"].fillna("DESCONHECIDO")
df_clean["Premis Cd"] = df_clean["Premis Cd"].astype("Int64")

# Weapon Used Cd / Weapon Desc / Mocodes ficam ausentes quando não se aplicam (ver regra 6).
print(df_clean["Vict Sex"].value_counts().to_string())

Vict Sex
M                374716
F                334355
NAO_APLICAVEL    120823
DESCONHECIDO      83753


In [11]:
# ---------- 6) Contagem de crimes e remoção de colunas redundantes ----------
crm_cols = ["Crm Cd 1", "Crm Cd 2", "Crm Cd 3", "Crm Cd 4"]
df_clean["Crm_Count"] = df_clean[crm_cols].notna().sum(axis=1)

df_clean = df_clean.drop(columns=crm_cols + ["LOCATION", "Cross Street", "Status Desc", "TIME OCC"])

# ---------- 7) Colunas derivadas ----------
df_clean["Week_OCC"] = df_clean["DATE OCC"].dt.day_name()

hora = df_clean["DATETIME OCC"].dt.hour
df_clean["shift_OCC"] = np.select(
    [hora.isna(), hora < 6, hora < 12, hora < 18],
    ["Unknown", "Dawn", "Morning", "Afternoon"],
    default="Night")

df_clean[["DATE OCC", "DATETIME OCC", "Week_OCC", "shift_OCC"]].head()

,DATE OCC,DATETIME OCC,Week_OCC,shift_OCC
0,2020-11-07,2020-11-07 08:45:00,Saturday,Morning
1,2020-10-18,2020-10-18 18:45:00,Sunday,Night
2,2020-10-30,2020-10-30 12:40:00,Friday,Afternoon
3,2020-12-24,2020-12-24 13:10:00,Thursday,Afternoon
4,2020-09-29,2020-09-29 18:30:00,Tuesday,Night


In [12]:
COLS_SAIDA = ["DR_NO", "Date Rptd", "DATE OCC", "DATETIME OCC", "Week_OCC", "shift_OCC",
              "AREA", "AREA NAME", "Rpt Dist No", "Part 1-2",
              "Crm Cd", "Crm Cd Desc", "Crm_Count", "Mocodes",
              "Vict Age", "Vict Sex", "Vict Descent",
              "Premis Cd", "Premis Desc", "Weapon Used Cd", "Weapon Desc",
              "Status", "LAT", "LON"]
df_clean = df_clean[COLS_SAIDA].reset_index(drop=True)

nulos = df_clean.isnull().sum()
print("Nulos remanescentes (esperados apenas onde o dado não se aplica):")
print(nulos[nulos > 0].to_string())

# Contrato com o notebook de clusterização: sem nulos nestas colunas
NAO_NULAS = ["DR_NO", "DATE OCC", "Crm Cd", "Crm Cd Desc", "Vict Sex", "Vict Descent",
             "Premis Desc", "Status", "AREA NAME"]
assert df_clean[NAO_NULAS].isnull().sum().sum() == 0
assert df_clean["DATE OCC"].max() <= pd.Timestamp(DATA_CORTE)
assert df_clean["DR_NO"].is_unique
print(f"\nBase limpa: {df_clean.shape[0]:,} linhas × {df_clean.shape[1]} colunas "
      f"({df_clean.shape[0] / n_inicial:.1%} da base bruta)")

Nulos remanescentes (esperados apenas onde o dado não se aplica):
Mocodes           127234
Vict Age          229906
Premis Cd             10
Weapon Used Cd    598432
Weapon Desc       598432
LAT                 2240
LON                 2240

Base limpa: 913,647 linhas × 24 colunas (90.9% da base bruta)


In [13]:
df_clean.to_csv(CAMINHO_LIMPO, index=False)
print(f"Salvo em {CAMINHO_LIMPO}. Próximo passo: notebook de clusterização.")

Salvo em data/processed/lapd_clean.csv. Próximo passo: notebook de clusterização.
